In [1]:
import pandas as pd

df = pd.read_json("hf://datasets/Rowden/CybersecurityQAA/cybersecurityQAAdataset.json")

In [4]:
row = df.iloc[0]

In [6]:
row["vars"]

{'question': "How can I ensure that my business's network access is restricted as per PCI DSS requirements?",
 'answer': "To ensure your business's network access is restricted according to PCI DSS requirements, you should ensure that inbound traffic to the Cardholder Data Environment (CDE) is restricted to only necessary traffic, and all other traffic is specifically denied. Similarly, outbound traffic from the CDE should also be restricted to necessary communications only. Implement Network Security Controls (NSCs) between all wireless networks and the CDE, and ensure that system components storing cardholder data are not directly accessible from untrusted networks. Additionally, anti-spoofing measures should be implemented to block forged IP addresses.",
 'evidence': '1.3.1 Inbound traffic to the CDE is restricted as follows: To only traffic that is necessary. All other traffic is specifically denied. 1.3.2 Outbound traffic from the CDE is restricted as follows: To only traffic that

In [7]:
row["assert"]

[{'type': 'llm-rubric',
  'value': 'The answer correctly provides steps to restrict network access in compliance with PCI DSS, as outlined in the evidence. This should include steps such as restricting network traffic to only that necessary, implementing network security controls between all wireless networks and the cardholder data environment, and ensure cardholder data is not accessible from untrusted networks.'}]

In [9]:
len(df)

1563

In [10]:
df.to_csv("/home/nam/projects/sid/RAG-Assignment3/data/hf_cyberqaa/raw/hf_cyberqaa_raw_df.csv")

#### Manually reading and chunking each raw data point into processed data point

In [1]:
import pandas as pd

df = pd.read_csv("/home/nam/projects/sid/RAG-Assignment3/data/hf_cyberqaa/raw/hf_cyberqaa_raw_df.csv")

In [14]:
from pathlib import Path
import json
import ast

output = []
missing_counts = {"question": 0, "answer": 0, "evidence": 0}

for idx, row in df.iterrows():
    # Defensive check: Ensure 'vars' exists and is a dictionary
    vars_data = ast.literal_eval(row.get('vars', {}))

    assertions = ast.literal_eval(row.get('assert', []))
    # print(type(assertions))
    # break
    eval_rubric = assertions[0].get('value', "") if assertions else "No rubric provided."

    # Extract fields and track missing ones
    q = vars_data.get('question')
    a = vars_data.get('answer')
    e = vars_data.get('evidence')

    # Log specific missing fields for your report
    if not q: missing_counts["question"] += 1
    if not a: missing_counts["answer"] += 1
    if not e: missing_counts["evidence"] += 1

    # Only create a chunk if we have the core information (Q and A)
    if q and a:
        combined_text = (
            f"QUESTION: {q}\n"
            f"ANSWER: {a}\n"
            f"EVIDENCE: {e if e else 'No specific evidence provided.'}"
        ).strip()
        
        chunk_entry = {
            "id": idx,
            "text": combined_text,
            "source": vars_data.get("sourceURL", "unknown"),
            "topic": vars_data.get("theme", "General"),
            "target_audience": vars_data.get("target_audience", "General"),
            "expert_score": vars_data.get("expert_score", 0),
            # --- EVALUATION SPECIFIC DATA ---
            "ground_truth": a,          # The gold answer for RAGAS
            "eval_rubric": eval_rubric,  # The criteria from the 'assert' block
            "original_evidence": e       # Pure evidence for context precision
        }
        output.append(chunk_entry)
    else:
        print(f"Index {idx} rejected: Missing critical Question or Answer.")

# Final report so you know exactly what happened
print("-" * 30)
print(f"PROCESSING COMPLETE")
print(f"Total rows processed: {len(df)}")
print(f"Successfully chunked: {len(output)}")
print(f"Rejected: {len(df) - len(output)}")
print(f"Missing Field Stats: {missing_counts}")
print("-" * 30)

# Only save if we actually got data
if output:
    output_path = Path("/home/nam/projects/sid/RAG-Assignment3/data/hf_cyberqaa/processed/hf_cyberqaa_chunks.json")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)

Index 319 rejected: Missing critical Question or Answer.
------------------------------
PROCESSING COMPLETE
Total rows processed: 1563
Successfully chunked: 1562
Rejected: 1
Missing Field Stats: {'question': 0, 'answer': 1, 'evidence': 0}
------------------------------


In [16]:
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# Load embedding model
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

# Initialize Qdrant
client = QdrantClient(path="/home/nam/projects/sid/RAG-Assignment3/qdrant_data")

COLLECTION_NAME = "hf_qaa_chunks"

# Create collection if not exists
collections = client.get_collections().collections
existing = [c.name for c in collections]

if COLLECTION_NAME not in existing:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=1024,
            distance=Distance.COSINE,
        ),
    )

# Load chunked data
input_path = Path("/home/nam/projects/sid/RAG-Assignment3/data/hf_cyberqaa/processed/hf_cyberqaa_chunks.json")

with open(input_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

points = []

for chunk in chunks:
    embedding = model.encode(chunk["text"]).tolist()

    points.append(
        PointStruct(
            id=chunk["id"],
            vector=embedding,
            payload={
                "text": chunk["text"],
                "source": chunk["source"],
                "topic": chunk["topic"],
                # Eval stuff
                "ground_truth": chunk.get("ground_truth"),
                "eval_rubric": chunk.get("eval_rubric"),
                "original_evidence": chunk.get("original_evidence")
            }
        )
    )

# Upload to Qdrant
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

print(f"Inserted {len(points)} chunks into Qdrant")
client.close()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inserted 1562 chunks into Qdrant


In [1]:
import requests
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

OLLAMA_URL = "http://localhost:11434/api/chat"

COLLECTION_NAME = "hf_qaa_chunks"

model = SentenceTransformer("BAAI/bge-large-en-v1.5")

client = QdrantClient(path="/home/nam/projects/sid/RAG-Assignment3/qdrant_data")

query = "Give me some information about NIST SP 800-53"

# Retrieve relevant chunks
query_embedding = model.encode(query).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=3,
).points

print("\nTOP RETRIEVED CHUNKS\n")
for idx, result in enumerate(results, start=1):
    print("=" * 80)
    print(f"Rank #{idx}")
    print(f"Score: {result.score:.4f}")
    print(f"Source: {result.payload['source']}")
    print()
    print(result.payload["text"])
    print()

client.close()

context = "\n\n".join(
    r.payload["text"]
    for r in results
)

prompt = f"""
You are a cybersecurity assistant.

Use ONLY the provided context to answer the user's question.

If the context does not contain the answer, say:
"I could not find sufficient information."

Provide concise and actionable cybersecurity guidelines.

CONTEXT:
{context}

QUESTION:
{query}
"""

payload = {
    "model": "gemma3:12b-it-qat",
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "stream": False
}

response = requests.post(OLLAMA_URL, json=payload)

data = response.json()

print("\nGENERATED ANSWER\n")
print(data["message"]["content"])
client.close()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


TOP RETRIEVED CHUNKS

Rank #1
Score: 0.7917
Source: https://doi.org/10.6028/NIST.SP.800-53r5

QUESTION: What is the purpose of NIST SP 800-53 Revision 5?
ANSWER: The purpose of NIST SP 800-53 Revision 5 is to provide a comprehensive set of security and privacy controls for information systems and organizations to protect against a wide range of threats and risks. It aims to help organizations manage security and privacy risks effectively, comply with applicable laws, and ensure the trustworthiness of their information technology products and systems.
EVIDENCE: This publication provides a catalog of security and privacy controls for information systems and organizations to protect organizational operations and assets, individuals, other organizations, and the Nation from a diverse set of threats and risks, including hostile attacks, human errors, natural disasters, structural failures, foreign intelligence entities, and privacy risks. The controls are flexible and customizable and impl